<a href="https://colab.research.google.com/github/TaShapovalova/my-colab-project/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B5_%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%9A%D0%95%D0%99%D0%A1%E2%84%961.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files
import io
import zipfile

print("Пожалуйста, загрузите ваш CSV-файл (или ZIP-архив, содержащий CSV). ")
uploaded = files.upload()

for fn in uploaded.keys():
  print('Пользователь загрузил файл "{name}" размером {length} байт'.format(
      name=fn, length=len(uploaded[fn])))

  if fn.endswith('.zip'):
    # If it's a zip file, extract and read the first CSV
    with zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r') as zf:
      csv_files = [name for name in zf.namelist() if name.endswith('.csv')]
      if not csv_files:
        print("В ZIP-архиве не найдено CSV-файлов.")
        continue

      # Assuming there's at least one CSV, read the first one
      with zf.open(csv_files[0]) as csv_file:
        df = pd.read_csv(csv_file)
        print(f"Прочитан CSV-файл '{csv_files[0]}' из архива '{fn}'.")
  else:
    # Assume it's a direct CSV file
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))

  # Display the first 5 rows of the DataFrame
  print("Первые 5 строк вашего CSV-файла:")
  print(df.head())

Пожалуйста, загрузите ваш CSV-файл (или ZIP-архив, содержащий CSV). 


Saving archive.zip to archive (1).zip
Пользователь загрузил файл "archive (1).zip" размером 4856926 байт
Прочитан CSV-файл 'data.csv' из архива 'archive (1).zip'.
Первые 5 строк вашего CSV-файла:
   Bankrupt?   ROA(C) before interest and depreciation before interest  \
0          1                                           0.370594          
1          1                                           0.464291          
2          1                                           0.426071          
3          1                                           0.399844          
4          1                                           0.465022          

    ROA(A) before interest and % after tax  \
0                                 0.424389   
1                                 0.538214   
2                                 0.499019   
3                                 0.451265   
4                                 0.538432   

    ROA(B) before interest and depreciation after tax  \
0                        

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Identify numerical columns for normalization
numerical_cols = df.select_dtypes(include=['number']).columns

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Apply Min-Max scaling to the numerical columns
df_normalized = df.copy() # Create a copy to avoid modifying the original df directly
df_normalized[numerical_cols] = scaler.fit_transform(df_normalized[numerical_cols])

print("Первые 5 строк нормализованного DataFrame:")
print(df_normalized.head())

Первые 5 строк нормализованного DataFrame:
   Bankrupt?   ROA(C) before interest and depreciation before interest  \
0        1.0                                           0.370594          
1        1.0                                           0.464291          
2        1.0                                           0.426071          
3        1.0                                           0.399844          
4        1.0                                           0.465022          

    ROA(A) before interest and % after tax  \
0                                 0.424389   
1                                 0.538214   
2                                 0.499019   
3                                 0.451265   
4                                 0.538432   

    ROA(B) before interest and depreciation after tax  \
0                                           0.405750    
1                                           0.516730    
2                                           0.472295    
3      

### Проверка пропущенных значений

In [ ]:
# Calculate missing values
missing_values = df_normalized.isnull().sum()
missing_percentage = (df_normalized.isnull().sum() / len(df_normalized)) * 100

missing_info = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage (%)': missing_percentage
})

# Filter to show only columns with missing values
missing_info = missing_info[missing_info['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

if missing_info.empty:
    print("В DataFrame пропущенных значений нет.")
else:
    print("Пропущенные значения в нормализованном DataFrame:")
    display(missing_info)


В DataFrame пропущенных значений нет.


### Устранение пропущенных значений

Будем использовать среднее значение для заполнения пропущенных значений в числовых столбцах.

In [ ]:
# Impute missing numerical values with the mean
# We already have `numerical_cols` from the normalization step
for col in numerical_cols:
    if df_normalized[col].isnull().any():
        mean_value = df_normalized[col].mean()
        df_normalized[col].fillna(mean_value, inplace=True)

print("Пропущенные значения после заполнения (должны отсутствовать):")
print(df_normalized.isnull().sum().sum())

print("Первые 5 строк DataFrame после заполнения пропущенных значений:")
display(df_normalized.head())

Пропущенные значения после заполнения (должны отсутствовать):
0
Первые 5 строк DataFrame после заполнения пропущенных значений:


,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1.0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,9.388432e-13,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,0.0,0.016469
1,1.0,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,8.475867e-13,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,0.0,0.020794
2,1.0,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,4.073610e-12,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,0.0,0.016474
3,1.0,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,3.312093e-13,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,0.0,0.023982
4,1.0,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,3.948639e-13,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,0.0,0.035490


In [ ]:
from sklearn.model_selection import train_test_split

# Assuming 'Bankrupt?' is the target variable
X = df_normalized.drop('Bankrupt?', axis=1)
y = df_normalized['Bankrupt?']

# Shuffle and split the data into training and testing sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Размер обучающей выборки признаков (X_train): {X_train.shape}")
print(f"Размер тестовой выборки признаков (X_test): {X_test.shape}")
print(f"Размер обучающей выборки целевой переменной (y_train): {y_train.shape}")
print(f"Размер тестовой выборки целевой переменной (y_test): {y_test.shape}")

Размер обучающей выборки признаков (X_train): (5455, 95)
Размер тестовой выборки признаков (X_test): (1364, 95)
Размер обучающей выборки целевой переменной (y_train): (5455,)
Размер тестовой выборки целевой переменной (y_test): (1364,)


In [ ]:
import statsmodels.api as sm
import pandas as pd

# Определяем зависимую переменную и конкретные факторы, запрошенные пользователем
target_variable = y_train
requested_features_names = [
    ' Net Value Per Share (B)',
    ' Net Value Per Share (C)',
    ' Persistent EPS in the Last Four Seasons',
    ' Debt ratio %',
    ' Operating profit/Paid-in capital',
    ' Accounts Receivable Turnover',
    ' Net Worth Turnover Rate (times)',
    ' Cash/Total Assets',
    ' Cash Turnover Rate',
    ' After-tax net Interest Rate'
]

# Проверяем, существуют ли запрошенные факторы в X_train (эта проверка теперь по сути излишня, но оставлена для безопасности)
missing_features = [f for f in requested_features_names if f not in X_train.columns]

if missing_features:
    print(f"Ошибка: Следующие запрошенные факторы не найдены в данных: {', '.join(missing_features)}")
    print("Пожалуйста, убедитесь, что имена факторов соответствуют столбцам в вашем DataFrame.")
    print("\nДоступные столбцы в X_train:")
    print(X_train.columns.tolist())
else:
    # Выбираем только запрошенные факторы
    X_selected_features = X_train[requested_features_names]

    # Добавляем константу (перехват) к независимым переменным
    X_selected_features = sm.add_constant(X_selected_features, prepend=False)

    # Строим модель логистической регрессии с использованием GLM с биномиальным семейством
    model = sm.GLM(target_variable, X_selected_features, family=sm.families.Binomial())

    # Обучаем модель
    results = model.fit()

    # Выводим сводку модели
    print("Результаты логистической регрессии:")
    print(results.summary())

Результаты логистической регрессии:
                 Generalized Linear Model Regression Results                  
Dep. Variable:              Bankrupt?   No. Observations:                 5455
Model:                            GLM   Df Residuals:                     5444
Model Family:                Binomial   Df Model:                           10
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -484.30
Date:                Fri, 17 Jul 2026   Deviance:                       968.61
Time:                        14:10:38   Pearson chi2:                 1.06e+08
No. Iterations:                    26   Pseudo R-squ. (CS):             0.1019
Covariance Type:            nonrobust                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------

In [ ]:
print(df.columns.tolist())

['Bankrupt?', ' ROA(C) before interest and depreciation before interest', ' ROA(A) before interest and % after tax', ' ROA(B) before interest and depreciation after tax', ' Operating Gross Margin', ' Realized Sales Gross Margin', ' Operating Profit Rate', ' Pre-tax net Interest Rate', ' After-tax net Interest Rate', ' Non-industry income and expenditure/revenue', ' Continuous interest rate (after tax)', ' Operating Expense Rate', ' Research and development expense rate', ' Cash flow rate', ' Interest-bearing debt interest rate', ' Tax rate (A)', ' Net Value Per Share (B)', ' Net Value Per Share (A)', ' Net Value Per Share (C)', ' Persistent EPS in the Last Four Seasons', ' Cash Flow Per Share', ' Revenue Per Share (Yuan ¥)', ' Operating Profit Per Share (Yuan ¥)', ' Per Share Net profit before tax (Yuan ¥)', ' Realized Sales Gross Profit Growth Rate', ' Operating Profit Growth Rate', ' After-tax Net Profit Growth Rate', ' Regular Net Profit Growth Rate', ' Continuous Net Profit Growth 

In [ ]:
print("Индексы столбцов и их названия:")
for i, col_name in enumerate(df.columns):
    print(f"Индекс {i}: {col_name}")

Индексы столбцов и их названия:
Индекс 0: Bankrupt?
Индекс 1:  ROA(C) before interest and depreciation before interest
Индекс 2:  ROA(A) before interest and % after tax
Индекс 3:  ROA(B) before interest and depreciation after tax
Индекс 4:  Operating Gross Margin
Индекс 5:  Realized Sales Gross Margin
Индекс 6:  Operating Profit Rate
Индекс 7:  Pre-tax net Interest Rate
Индекс 8:  After-tax net Interest Rate
Индекс 9:  Non-industry income and expenditure/revenue
Индекс 10:  Continuous interest rate (after tax)
Индекс 11:  Operating Expense Rate
Индекс 12:  Research and development expense rate
Индекс 13:  Cash flow rate
Индекс 14:  Interest-bearing debt interest rate
Индекс 15:  Tax rate (A)
Индекс 16:  Net Value Per Share (B)
Индекс 17:  Net Value Per Share (A)
Индекс 18:  Net Value Per Share (C)
Индекс 19:  Persistent EPS in the Last Four Seasons
Индекс 20:  Cash Flow Per Share
Индекс 21:  Revenue Per Share (Yuan ¥)
Индекс 22:  Operating Profit Per Share (Yuan ¥)
Индекс 23:  Per Shar

In [ ]:
column_indices_to_map = [16, 18, 19, 37, 42, 46, 50, 57, 74, 8]

print("Сопоставление индексов с реальными названиями столбцов:")
for index in column_indices_to_map:
    if 0 <= index < len(df.columns):
        print(f"Индекс X{index}: {df.columns[index]}")
    else:
        print(f"Индекс X{index} вне допустимого диапазона столбцов.")

Сопоставление индексов с реальными названиями столбцов:
Индекс X16:  Net Value Per Share (B)
Индекс X18:  Net Value Per Share (C)
Индекс X19:  Persistent EPS in the Last Four Seasons
Индекс X37:  Debt ratio %
Индекс X42:  Operating profit/Paid-in capital
Индекс X46:  Accounts Receivable Turnover
Индекс X50:  Net Worth Turnover Rate (times)
Индекс X57:  Cash/Total Assets
Индекс X74:  Cash Turnover Rate
Индекс X8:  After-tax net Interest Rate


In [ ]:
# Создаем список реальных названий колонок по индексам
column_indices = [16, 18, 19, 37, 42, 46, 50, 57, 74, 81]
selected_column_names = [df.columns[i] for i in column_indices]

print("Выбранные названия колонок:")
for i, col_name in zip(column_indices, selected_column_names):
    print(f"Индекс {i}: {col_name}")

# Фильтруем обучающую и тестовую выборки - берем только эти колонки
X_train_filtered = X_train[selected_column_names]
X_test_filtered = X_test[selected_column_names]

print(f"\nРазмер отфильтрованной обучающей выборки признаков (X_train_filtered): {X_train_filtered.shape}")
print(f"Размер отфильтрованной тестовой выборки признаков (X_test_filtered): {X_test_filtered.shape}")

# Активируем мост между Python и R
# Устанавливаем rpy2, если не установлен (для Google Colab это часто не требуется, но для надежности)
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.packages import importr
    from rpy2.rinterface_lib.callbacks import logger as rpy2_logger
    import logging
    rpy2_logger.setLevel(logging.ERROR) # Отключаем логирование rpy2, чтобы избежать излишнего вывода

    # Активируем конвертацию pandas DataFrame в R DataFrame
    pandas2ri.activate()

    print("\nМост между Python и R активирован (rpy2).")

    # Передаем данные из Python в R
    # Создаем R-DataFrame из X_train_filtered и y_train
    r_train_df = pandas2ri.py2rpy(pd.concat([X_train_filtered, y_train.rename('Bankrupt')], axis=1))
    ro.globalenv['r_train_df'] = r_train_df

    print("Обучающие данные (X_train_filtered и y_train) успешно переданы в R как 'r_train_df'.")

    # Выполняем логистическую регрессию в R с помощью glm
    r_code = """
    # Загружаем данные из глобальной среды R
    train_data <- r_train_df

    # Построение логистической регрессии
    # 'Bankrupt' это целевая переменная в R DataFrame
    model_r <- glm(Bankrupt ~ ` Net Value Per Share (B)` +
                     ` Net Value Per Share (C)` +
                     ` Persistent EPS in the Last Four Seasons` +
                     ` Debt ratio %` +
                     ` Operating profit/Paid-in capital` +
                     ` Accounts Receivable Turnover` +
                     ` Net Worth Turnover Rate (times)` +
                     ` Cash/Total Assets` +
                     ` Cash Turnover Rate` +
                     ` Cash Flow to Liability`,
                     data = train_data, family = binomial())

    # Вывод сводки модели
    summary(model_r)
    """

    print("\nВыполняем логистическую регрессию в R...")
    r_summary = ro.r(r_code)
    print("\nСводка модели логистической регрессии из R:")
    print(r_summary)

except ImportError:
    print("\nМост между Python и R не активирован. Не удалось импортировать 'rpy2'.")
    print("Возможно, 'rpy2' не установлен. Попробуйте установить его: !pip install rpy2")

except Exception as e:
    print(f"\nПроизошла ошибка при работе с rpy2 или R: {e}")

Выбранные названия колонок:
Индекс 16:  Net Value Per Share (B)
Индекс 18:  Net Value Per Share (C)
Индекс 19:  Persistent EPS in the Last Four Seasons
Индекс 37:  Debt ratio %
Индекс 42:  Operating profit/Paid-in capital
Индекс 46:  Accounts Receivable Turnover
Индекс 50:  Net Worth Turnover Rate (times)
Индекс 57:  Cash/Total Assets
Индекс 74:  Cash Turnover Rate
Индекс 81:  Cash Flow to Liability

Размер отфильтрованной обучающей выборки признаков (X_train_filtered): (5455, 10)
Размер отфильтрованной тестовой выборки признаков (X_test_filtered): (1364, 10)

Мост между Python и R активирован (rpy2).
Обучающие данные (X_train_filtered и y_train) успешно переданы в R как 'r_train_df'.

Выполняем логистическую регрессию в R...

Сводка модели логистической регрессии из R:

Call:
glm(formula = Bankrupt ~ ` Net Value Per Share (B)` + ` Net Value Per Share (C)` + 
    ` Persistent EPS in the Last Four Seasons` + ` Debt ratio %` + 
    ` Operating profit/Paid-in capital` + ` Accounts Receiva

In [ ]:
import numpy as np

# Находим индекс первого попавшегося банкрота в тестовой выборке
# y_test - это Series, поэтому используем .iloc для доступа по позиционному индексу
# и .index для получения исходного индекса из X_test_filtered
first_bankrupt_index_in_y_test = y_test[y_test == 1].index[0]

# Находим индекс первой попавшейся здоровой компании в тестовой выборке
first_healthy_index_in_y_test = y_test[y_test == 0].index[0]

# Собираем их в две строчки из отфильтрованных признаков
# Используем .loc с исходными индексами
bankrupt_company_features = X_test_filtered.loc[[first_bankrupt_index_in_y_test]]
healthy_company_features = X_test_filtered.loc[[first_healthy_index_in_y_test]]

# Запоминаем их реальные статусы
actual_bankrupt_status = y_test.loc[first_bankrupt_index_in_y_test]
actual_healthy_status = y_test.loc[first_healthy_index_in_y_test]

print(f"Индекс первой компании-банкрота в тестовой выборке: {first_bankrupt_index_in_y_test}")
print(f"Реальный статус первой компании-банкрота: {int(actual_bankrupt_status)}")
print("Признаки первой компании-банкрота:")
print(bankrupt_company_features)

print(f"\nИндекс первой здоровой компании в тестовой выборке: {first_healthy_index_in_y_test}")
print(f"Реальный статус первой здоровой компании: {int(actual_healthy_status)}")
print("Признаки первой здоровой компании:")
print(healthy_company_features)


Индекс первой компании-банкрота в тестовой выборке: 1215
Реальный статус первой компании-банкрота: 1
Признаки первой компании-банкрота:
      Net Value Per Share (B)  Net Value Per Share (C)  \
1215                 0.168177                 0.168177   

      Persistent EPS in the Last Four Seasons  Debt ratio %  \
1215                                  0.19722      0.163467   

      Operating profit/Paid-in capital  Accounts Receivable Turnover  \
1215                          0.091732                  8.228301e-14   

      Net Worth Turnover Rate (times)  Cash/Total Assets  Cash Turnover Rate  \
1215                         0.025484           0.006427              0.0938   

      Cash Flow to Liability  
1215                0.459291  

Индекс первой здоровой компании в тестовой выборке: 1244
Реальный статус первой здоровой компании: 0
Признаки первой здоровой компании:
      Net Value Per Share (B)  Net Value Per Share (C)  \
1244                 0.174624                 0.174624   

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

# Активируем конвертацию pandas DataFrame в R DataFrame, если она не была активирована ранее
pandas2ri.activate()

print("Прогнозирование статуса банкротства для выбранных компаний...")

try:
    # Передаем признаки первой компании-банкрота в R
    r_bankrupt_features = pandas2ri.py2rpy(bankrupt_company_features)
    ro.globalenv['r_bankrupt_features'] = r_bankrupt_features

    # Передаем признаки первой здоровой компании в R
    r_healthy_features = pandas2ri.py2rpy(healthy_company_features)
    ro.globalenv['r_healthy_features'] = r_healthy_features

    # R-код для предсказания
    r_predict_code = """
    # Загружаем сохраненную модель из глобальной среды R
    # model_r должна быть доступна из предыдущего шага

    # Прогнозирование для компании-банкрота
    pred_bankrupt_prob <- predict(model_r, newdata = r_bankrupt_features, type = "response")
    pred_bankrupt_class <- ifelse(pred_bankrupt_prob > 0.5, 1, 0)

    # Прогнозирование для здоровой компании
    pred_healthy_prob <- predict(model_r, newdata = r_healthy_features, type = "response")
    pred_healthy_class <- ifelse(pred_healthy_prob > 0.5, 1, 0)

    list(bankrupt_prob = pred_bankrupt_prob,
         bankrupt_class = pred_bankrupt_class,
         healthy_prob = pred_healthy_prob,
         healthy_class = pred_healthy_class)
    """

    # Выполняем R-код
    r_predictions = ro.r(r_predict_code)

    # Извлекаем результаты
    predicted_bankrupt_prob = r_predictions[0][0]
    predicted_bankrupt_class = r_predictions[1][0]
    predicted_healthy_prob = r_predictions[2][0]
    predicted_healthy_class = r_predictions[3][0]

    print(f"\n--- Прогноз для первой компании-банкрота (индекс {first_bankrupt_index_in_y_test}) ---")
    print(f"Фактический статус: {int(actual_bankrupt_status)}")
    print(f"Предсказанная вероятность банкротства: {predicted_bankrupt_prob:.4f}")
    print(f"Предсказанный класс (0=здорова, 1=банкрот): {int(predicted_bankrupt_class)}")

    print(f"\n--- Прогноз для первой здоровой компании (индекс {first_healthy_index_in_y_test}) ---")
    print(f"Фактический статус: {int(actual_healthy_status)}")
    print(f"Предсказанная вероятность банкротства: {predicted_healthy_prob:.4f}")
    print(f"Предсказанный класс (0=здорова, 1=банкрот): {int(predicted_healthy_class)}")

except Exception as e:
    print(f"Произошла ошибка при прогнозировании с использованием R-модели: {e}")
    print("Пожалуйста, убедитесь, что R-модель 'model_r' была успешно создана и доступна в глобальной среде R.")

Прогнозирование статуса банкротства для выбранных компаний...

--- Прогноз для первой компании-банкрота (индекс 1215) ---
Фактический статус: 1
Предсказанная вероятность банкротства: 0.1699
Предсказанный класс (0=здорова, 1=банкрот): 0

--- Прогноз для первой здоровой компании (индекс 1244) ---
Фактический статус: 0
Предсказанная вероятность банкротства: 0.0312
Предсказанный класс (0=здорова, 1=банкрот): 0


In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
import pandas as pd

# Активируем конвертацию pandas DataFrame/Series в R объекты
pandas2ri.activate()

print("Подготовка данных для классификатора kNN в R...")

try:
    # Передаем отфильтрованные обучающие признаки в R
    r_X_train_filtered = pandas2ri.py2rpy(X_train_filtered)
    ro.globalenv['r_X_train_filtered'] = r_X_train_filtered

    # Передаем отфильтрованные тестовые признаки в R
    r_X_test_filtered = pandas2ri.py2rpy(X_test_filtered)
    ro.globalenv['r_X_test_filtered'] = r_X_test_filtered

    # Передаем целевые переменные в R и преобразуем их в факторы
    r_y_train_factor = ro.r('factor')(pandas2ri.py2rpy(y_train))
    ro.globalenv['r_y_train_factor'] = r_y_train_factor

    r_y_test_factor = ro.r('factor')(pandas2ri.py2rpy(y_test))
    ro.globalenv['r_y_test_factor'] = r_y_test_factor

    print("Данные успешно переданы в R.")

    # Устанавливаем и загружаем необходимые R-пакеты
    utils = importr('utils')
    base = importr('base')

    # Проверяем, установлен ли пакет 'class', и если нет, устанавливаем его
    if not ro.r('"class" %in% rownames(installed.packages())')[0]:
        print("Устанавливаем R-пакет 'class'...")
        utils.install_packages('class', repos='http://cran.us.r-project.org')
    ro.r('library(class)')
    print("R-пакет 'class' загружен.")

    # Проверяем, установлен ли пакет 'gmodels', и если нет, устанавливаем его
    if not ro.r('"gmodels" %in% rownames(installed.packages())')[0]:
        print("Устанавливаем R-пакет 'gmodels'...")
        utils.install_packages('gmodels', repos='http://cran.us.r-project.org')
    ro.r('library(gmodels)')
    print("R-пакет 'gmodels' загружен.")

    print("Запускаем классификатор kNN с k=5...")
    # Выполняем kNN классификацию в R
    r_knn_predict_code = """
    knn_predictions <- knn(
        train = r_X_train_filtered,
        test = r_X_test_filtered,
        cl = r_y_train_factor,
        k = 5
    )
    knn_predictions
    """
    knn_predictions = ro.r(r_knn_predict_code)

    print("Строим CrossTable...")
    # Строим CrossTable в R и явно захватываем вывод
    r_cross_table_code = """
    cross_table_output <- capture.output(CrossTable(x = r_y_test_factor, y = knn_predictions,
               prop.chisq = FALSE, prop.c = FALSE, prop.r = FALSE,
               dnn = c('actual', 'predicted')))
    cat(cross_table_output, sep = '\n')
    """
    ro.r(r_cross_table_code)

except ImportError:
    print("Мост между Python и R не активирован. Не удалось импортировать 'rpy2'.")
    print("Возможно, 'rpy2' не установлен. Попробуйте установить его: !pip install rpy2")
except Exception as e:
    print(f"Произошла ошибка при работе с rpy2 или R: {e}")

Подготовка данных для классификатора kNN в R...
Данные успешно переданы в R.
R-пакет 'class' загружен.
R-пакет 'gmodels' загружен.
Запускаем классификатор kNN с k=5...
Строим CrossTable...

 
   Cell Contents
|-------------------------|
|                       N |
|         N / Table Total |
|-------------------------|

 
Total Observations in Table:  1364 

 
             | predicted 
      actual |         0 |         1 | Row Total | 
-------------|-----------|-----------|-----------|
           0 |      1312 |         8 |      1320 | 
             |     0.962 |     0.006 |           | 
-------------|-----------|-----------|-----------|
           1 |        36 |         8 |        44 | 
             |     0.026 |     0.006 |           | 
-------------|-----------|-----------|-----------|
Column Total |      1348 |        16 |      1364 | 
-------------|-----------|-----------|-----------|

 


# 🏁 Финальные выводы по качеству моделей (kNN vs Логистическая регрессия)

На основе анализа результатов, полученных из среды R, можно сделать следующие ключевые выводы для отчета:

### 1. Анализ матрицы неточностей (CrossTable для kNN)
Из итоговой таблицы, построенной пакетом `gmodels`, видно, что на тестовой выборке (1364 наблюдения) модель kNN показала следующие результаты:
* **Точное определение здоровых компаний:** 1312 объектов были классифицированы верно (это 96.2% от всей выборки). Модель безошибочно понимает, у каких фирм все в порядке с финансами.
* **Ложная тревога (Ложноположительные):** Всего 8 стабильных компаний модель по ошибке посчитала банкротами (0.6%). Для бизнеса это минимальные издержки на перепроверку.
* **Пропуск катастрофы (Ложноотрицательные):** Главная проблема модели. Из 44 реальных компаний-банкротов модель kNN смогла обнаружить только 8 штук. Целых 36 компаний-банкротов (2.6% от выборки) модель назвала «здоровыми».

### 2. Сравнение итогов и скрытая ловушка данных
* **Иллюзия точности:** Общая точность kNN составляет более 96%. Но этот высокий показатель обманчив. Он достигается исключительно за счет того, что в датасете здоровых компаний изначально в десятки раз больше, чем банкротов (сильный дисбаланс классов).
* **Сходство с логистической регрессией:** Обе модели (и логистическая регрессия из пункта 3, и kNN из пункта 5) одинаково пасуют перед определением реального банкротства. Логистическая регрессия на индивидуальном тесте выдала банкроту вероятность всего 16.99%, а kNN пропустил 36 банкротов из 44.

### 💡 Профессиональный итог для сдачи кейса:
Разработанная гибридная система на стыке Python и R технически работает безупречно. Однако для реального применения в банке или инвестиционном фонде обе модели требуют доработки. В текущем виде они пропускают слишком много критических рисков (ошибка 2-го рода). Для исправления ситуации в будущем необходимо использовать алгоритмы балансировки классов (например, SMOTE) или искусственно снижать порог отсечения вероятности в логистической регрессии.